In [ ]:
import ipywidgets as widgets

In [ ]:
from copy import deepcopy
from decimal import Decimal
import itertools
from pathlib import Path
from typing import cast, TYPE_CHECKING

from rosbag2_py import StorageFilter
from rosbag2_py import StorageOptions

from thesis_data_processing import CONTROLLER_MANAGER_DIAGNOSTIC_NAME_MAPPING
from thesis_data_processing import DataConsistencyChecker
from thesis_data_processing import DiagnosticsCollector
from thesis_data_processing import open_rosbag
from thesis_data_processing import read_messages
from thesis_data_processing import StatisticsCollector
from thesis_data_processing import SYSTEM_DIAGNOSTIC_NAME_MAPPING
from thesis_data_processing import utils
from thesis_data_processing.step_response import INITIAL_COMMAND_KEY
from thesis_data_processing.step_response import STEP_COMMAND_KEY
from thesis_data_processing.step_response import T_STEP_KEY

if TYPE_CHECKING:
    from typing import Optional

    from builtin_interfaces.msg import Time as MsgTime
    from controller_manager_msgs.msg import ControllerManagerActivity
    from controller_manager_msgs.msg import NamedLifecycleState
    from rosbag2_py import BagMetadata


In [ ]:
datachecker = DataConsistencyChecker(
    excluded_keys=('recording_duration', 'recording_date', STEP_COMMAND_KEY),
)

diagnostics_name_mapping = {}
diagnostics_name_mapping.update(CONTROLLER_MANAGER_DIAGNOSTIC_NAME_MAPPING)
diagnostics_name_mapping.update(SYSTEM_DIAGNOSTIC_NAME_MAPPING)

In [ ]:
rosbag_folder = Path('~/thesis/measurements') / 'experimental' / 'floor2'
rosbag_folder /= 'step-mmt-R20-2025-10-23T1032'
# rosbag_folder /= 'step-mmt-R10-2025-10-23T1102'
# rosbag_folder /= 'step-srv-R20-2025-10-23T1433'
# rosbag_folder /= 'step-srv-R10-2025-10-23T1411'

rosbag_folder = rosbag_folder.expanduser().resolve()

assert rosbag_folder.exists(), rosbag_folder

print('Folder:', rosbag_folder)

In [ ]:
rosbag_paths = sorted(rosbag_folder.glob('*'))

recording_selector = widgets.Select(
    description='recording:',
    # label='kaas',
    options=[(p.stem,p) for p in rosbag_paths],
    value=rosbag_paths[-2],
    layout={'width': 'max-content'},
    rows=5,
)

display(recording_selector)

In [ ]:
rosbag_path = recording_selector.value

print('Rosbag:', rosbag_path)

# TODO(SuperJappie08): Do something with diagnostics
storage_filter = StorageFilter(
    topics=[
        '/controller_manager/introspection_data/names',
        '/controller_manager/introspection_data/values',
        '/controller_manager/activity',
        '/diagnostics',
        '/rosout',
    ],
    regex_to_exclude='.*/_service_event',
)

wheel_names: set[str] = {
    f'{fb_pos}_{side}_wheel_joint'
    for fb_pos, side in itertools.product(('front', 'rear'), ('left', 'right'))
}

state_interfaces: set[str] = {
    f'state_interface.{wheel_name}/velocity'
    for wheel_name in wheel_names
}

command_interfaces: set[str] = {
    f'command_interface.{wheel_name}/velocity'
    for wheel_name in wheel_names
}

names_to_keep: set[str] = state_interfaces | command_interfaces


In [ ]:
with open_rosbag(StorageOptions(uri=str(rosbag_path))) as reader:
    metadata: 'BagMetadata' = reader.get_metadata()

    custom_metadata = deepcopy(metadata.custom_data)
    custom_metadata['ros_distro'] = metadata.ros_distro

    assert datachecker.check(custom_metadata), f"Bag '{rosbag_path.stem}' is inconsistent!"

    step_command = Decimal(custom_metadata[STEP_COMMAND_KEY])
    t_step = Decimal(custom_metadata[T_STEP_KEY])
    initial_command = Decimal(custom_metadata[INITIAL_COMMAND_KEY])

    controller_name = custom_metadata.get('controller_name', 'multi_wheel_step_controller')

    statistics_collector = StatisticsCollector(
        '/controller_manager/introspection_data',
        only_names=names_to_keep,
    )

    diagnostics_collector = DiagnosticsCollector(
        name_mapping=diagnostics_name_mapping,
    )

    start_activity_time: 'Optional[MsgTime]' = None
    accepting_data: bool = False
    for topic, msg, recv_time in read_messages(reader, storage_filter):
        if (
            topic == '/rosout'
            and msg.name == 'controller_manager'
            and msg.msg == f'Activating controllers: [ {controller_name} ]'
        ):
            start_activity_time = msg.stamp
        elif topic.endswith('/activity'):
            controller_status: 'NamedLifecycleState' = next(
                filter(
                    lambda controller: controller.name == controller_name,
                    cast('ControllerManagerActivity', msg).controllers,
                ),
            )

            # TODO: Maybe do this the time stamps instead to prevent different ordering
            accepting_data = controller_status.state.id == utils.LIFECYCLE_ACTIVE_ID
            # logger.info(
            #     "%s recording data on '%s'",
            #     'Started' if accepting_data else 'Stopped',
            #     statistics_collector.base_topic,
            # )
        elif topic.startswith(statistics_collector.base_topic) and (
            accepting_data or topic.endswith('/names')
        ):
            statistics_collector.process_msg(topic, msg, try_process=False)
        elif accepting_data and topic == '/diagnostics':
            diagnostics_collector.process_msg(msg, try_process=False)

    assert start_activity_time is not None

In [ ]:
bag_df = statistics_collector.data.copy(True)
diagnostics_df = diagnostics_collector.data.copy(True)
# First attempt to synchronize based on activation
bag_df.index = bag_df.index - utils.as_time(start_activity_time)
diagnostics_df.index = diagnostics_df.index - utils.as_time(start_activity_time)

# NOTE: In order to average multiple measurements it is necessary to index them
#       exactly. Therefore we assume that the first reported
#       To achieve this its assumed all command velocities are the equal.
some_command_interface = next(iter(command_interfaces))
assert (bag_df[list(command_interfaces)]).eq(
    bag_df.loc[:, some_command_interface], axis=0,
).all(1).all(), 'All command signals should have an equal t_step'

measured_command_t_step = \
    bag_df[bag_df[some_command_interface] != initial_command].index[0]
bag_df.index = bag_df.index - measured_command_t_step + t_step
bag_df.index = [
    abs(round(idx, 2)) if round(idx, 2).is_zero() else round(idx, 2)
    for idx in bag_df.index
]

diagnostics_df.index = diagnostics_df.index - measured_command_t_step + t_step
diagnostics_df.index = [
    abs(round(idx, 2)) if round(idx, 2).is_zero() else round(idx, 2)
    for idx in diagnostics_df.index
]

In [ ]:
diagnostics_df[[
    'Mirte-867B16.cpu-monitor.CPU Load Average',
    'Mirte-867B16.ram-monitor.RAM Load Average',
]].dropna(how='all')

In [ ]:
diagnostics_df[[
    'ros2_control.cm-cm-activity.periodicity.max',
    'ros2_control.cm-cm-activity.periodicity.min',
    'ros2_control.cm-cm-activity.periodicity.average',
    'ros2_control.cm-cm-activity.periodicity.standard_deviation',
]].dropna(how='all')


In [ ]:
columns = [column for column in diagnostics_df.columns if column.startswith('ros2_control.cm-cm')]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [column for column in diagnostics_df.columns if column.startswith('ros2_control.cm-cr')]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [column for column in diagnostics_df.columns if column.startswith('ros2_control.cm-hw-activity') and column.count('.') == 2]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [
    column
    for column in diagnostics_df.columns
    if column.startswith('ros2_control.cm-hw-activity')
    and (not column.endswith('state'))
    and ('arm' in column or 'gripper' in column)
]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [
    column
    for column in diagnostics_df.columns
    if column.startswith('ros2_control.cm-hw-activity')
    and (not column.endswith('state'))
    and (not column.startswith('ros2_control.cm-hw-activity.ros2_'))
]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
display(diagnostics_df.columns)

In [ ]:
diagnostics_df